# Weak-lensing galaxy shape catalogue validation

## Local metacalibration

Contents.
- Metacalibration (local)

In [ ]:
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

from sp_validation.io import *

In [ ]:
sp_base= f"{os.environ['HOME']}/astro/repositories/github/sp_validation"

# The following commands will be replaced by import instructions, once the sp_validation scripts are stable
for sc in ['io', 'galaxy']:
    script = os.path.join(sp_base, 'sp_validation', sc)
    %run $script

### Spatial binning

In [ ]:
# Projection all objects from spherical to Cartesian coordinates
x, y =  radec2xy(np.mean(ra_ngmix), np.mean(dec_ngmix), ra_ngmix, dec_ngmix)
#x, y = radec2xy(np.mean(dd['XWIN_WORLD']), np.mean(dd['YWIN_WORLD']), dd['XWIN_WORLD'], dd['YWIN_WORLD'])

#### Compute and enlarge field size

In [ ]:
# Define mix, max and size
min_x = np.min(x)
max_x = np.max(x)
min_y = np.min(y)
max_y = np.max(y)

size_x = max_x - min_x
size_y = max_y - min_y
size_x_deg = np.rad2deg(size_x)
size_y_deg = np.rad2deg(size_y)

print(f'Field size in projected coordinates is (x, y) = ({size_x_deg:.2f}, {size_y_deg:.2f}) deg')

In [ ]:
# Calibration pixel sizes for local calibration, in degree
cal_pix_size_deg = [1, 2, 4]

In [ ]:
# Compute larger field size, divisible by largest subfield size
factor = max(cal_pix_size_deg)

size_x_deg_new = np.ceil(size_x_deg / factor) * factor
size_y_deg_new = np.ceil(size_y_deg / factor) * factor

print(f'Enlarged field size in projected coordinates is (x, y) = ({size_x_deg_new:.1f}, {size_y_deg_new:.1f}) deg')

In [ ]:
# Modify max coordinates to account for enlarged field
max_x_new = max_x + np.deg2rad(size_x_deg_new - size_x_deg)
max_y_new = max_y + np.deg2rad(size_y_deg_new - size_y_deg)
size_x_new_check = max_x_new - min_x
size_y_new_check = max_y_new - min_y

# Check that new size is ok
print('Check new field size:', np.rad2deg(size_x_new_check), np.rad2deg(size_y_new_check))

#### Local calibration

In [ ]:
def local_calib(dd, x, y, min_x, max_x, min_y, max_y, npix_x, npix_y, R_shear_moy, R_selec_moy, dm, sigm, stats_file, verbose=False):
    """Local Calib
    
    Local calibration.
    """
    
    # Mean total calibration matrix
    R_tot_moy = R_shear_moy + R_selec_moy
    
    g1_final = np.array([])
    g2_final = np.array([])
    g1_final_dm = np.array([])
    g2_final_dm = np.array([])
    R_selec = np.zeros((2, 2, npix_y, npix_x))
    R_shear = np.zeros((2, 2, npix_y, npix_x))
    ra_ngmix2 = np.array([])
    dec_ngmix2 = np.array([])
    R_selec_std = np.zeros((2, 2, npix_y, npix_x))
    R_shear_std = np.zeros((2, 2, npix_y, npix_x))
    c1_ngmix_cut = np.zeros((npix_y, npix_x))
    c2_ngmix_cut = np.zeros((npix_y, npix_x))
    err_c1_ngmix_cut = np.zeros((npix_y, npix_x))
    err_c2_ngmix_cut = np.zeros((npix_y, npix_x))
    
    # MKDEBUG was xx, yy
    ngal = bin2d(x, y, npix=(npix_x, npix_y), extent=(min_x, max_x, min_y, max_y))

    # Cut the catalogue into calibration pixels -> dd_cut
    for i in range(npix_x):
        for j in range(npix_y):
            if i == npix_x and j != npix_y:
                dd_cut = dd[np.where((x >= min_x + (i*size_x/npix_x)) & (x <= max_x) \
                                     & (y >= min_y + (j*size_y/npix_y)) & (y < min_y + ((j+1)*size_y/npix_y)))]
            elif j== npix_x and i != npix_y:
                dd_cut = dd[np.where((x >= min_x + (i*size_x/npix_x)) & (x < min_x + ((i+1)*size_x/npix_x)) \
                                     & (y >= min_y + (j*size_y/npix_y)) & (y <= max_y))] 
            elif j == npix_x and i == npix_y:
                dd_cut = dd[np.where((x >= min_x + (i*size_x/npix_x)) & (x <= max_x) \
                                     & (y >= min_y + (j*size_y/npix_y)) & (y <= max_y ))]
            else:
                dd_cut = dd[np.where((x >= min_x + (i*size_x/npix_x)) & (x < min_x + ((i+1)*size_x/npix_x)) \
                                     & (y >= min_y + (j*size_y/npix_y)) & (y < min_y + ((j+1)*size_y/npix_y)))]
        
            # Mask of dd_cut
            cut_common = classification_galaxy_base(dd_cut)
            m_gal_ngmix_cut = classification_galaxy_ngmix(dd_cut, cut_common, stats_file, verbose=verbose)
 

            # Apply metacal to dd_cut
            gal_metacal_ngmix_cut = metacal(dd_cut, m_gal_ngmix_cut, verbose=verbose)
        
            # Save ra & dec to keep the order        
            ra_ngmix_temp = dd_cut['XWIN_WORLD'][m_gal_ngmix_cut][gal_metacal_ngmix_cut.mask_dict['ns']]
            ra_ngmix2 = np.concatenate((ra_ngmix2, ra_ngmix_temp))
            dec_ngmix_temp = dd_cut['YWIN_WORLD'][m_gal_ngmix_cut][gal_metacal_ngmix_cut.mask_dict['ns']]
            dec_ngmix2 = np.concatenate((dec_ngmix2, dec_ngmix_temp))
            
            w_ngmix_cut = gal_metacal_ngmix_cut.ns['w'][gal_metacal_ngmix_cut.mask_dict['ns']]
        
            g_ngmix_cut = np.array([gal_metacal_ngmix_cut.ns['g1'][gal_metacal_ngmix_cut.mask_dict['ns']], gal_metacal_ngmix_cut.ns['g2'][gal_metacal_ngmix_cut.mask_dict['ns']]])
        
            # If numver of galaxy is ok, metacal local
            if ngal[j,i] > np.mean(ngal) / 2:
                g_corr_ngmix_cut = np.linalg.inv(gal_metacal_ngmix_cut.R).dot(g_ngmix_cut)
                
                # Add of delta m 
                R_dm = gal_metacal_ngmix_cut.R + np.ones((2,2)) * (dm + np.random.normal(0, sigm))
                g_corr_ngmix_cut_dm = np.linalg.inv(R_dm).dot(g_ngmix_cut)
                
                # Additive bias
                c1_ngmix_cut[j,i], err_c1_ngmix_cut[j,i] = jackknif_weighted_average2(g_corr_ngmix_cut[0], w_ngmix_cut, remove_size=0.05, n_realization=500)
                c2_ngmix_cut[j,i], err_c2_ngmix_cut[j,i] = jackknif_weighted_average2(g_corr_ngmix_cut[1], w_ngmix_cut, remove_size=0.05, n_realization=500)
                
                # Save R matrix and std
                R_selec[:,:,j,i] = gal_metacal_ngmix_cut.R_selection
                R_shear[:,:,j,i] = np.mean(gal_metacal_ngmix_cut.R_shear,2)
                #R_selec_std[:,:,j,i] = gal_metacal_ngmix_cut.R_selection_std
                #R_shear_std[:,:,j,i] = gal_metacal_ngmix_cut.R_shear_std
                                
            # If low number of galaxy, we use value of globaal metacal
            else:
                g_corr_ngmix_cut = np.linalg.inv(R_tot_moy).dot(g_ngmix_cut)
                R_dm = R_tot_moy + np.ones((2,2)) * (dm + np.random.normal(0, sigm))
                g_corr_ngmix_cut_dm = np.linalg.inv(R_dm).dot(g_ngmix_cut)
                
                # MKDEBUG: Changed the following from mean values to zero
                c1_ngmix_cut[j,i] = 0
                err_c1_ngmix_cut[j,i] = 0
                c2_ngmix_cut[j,i] = 0
                err_c2_ngmix_cut[j,i] = 0
            
                R_selec[:,:,j,i] = R_selec_moy
                R_shear[:,:,j,i] = R_shear_moy
                #R_selec_std[:,:,j,i] = R_selec_moy_std
                #R_shear_std[:,:,j,i] = R_shear_moy_std
                             
            g1_final = np.concatenate((g1_final, g_corr_ngmix_cut[0]))  
            g2_final = np.concatenate((g2_final, g_corr_ngmix_cut[1]))
            g1_final_dm = np.concatenate((g1_final_dm, g_corr_ngmix_cut_dm[0]))  
            g2_final_dm = np.concatenate((g2_final_dm, g_corr_ngmix_cut_dm[1]))
    
    return (
        np.array([g1_final, g2_final]), R_shear, R_selec, ra_ngmix2, dec_ngmix2,
        np.array([c1_ngmix_cut, c2_ngmix_cut]), np.array([err_c1_ngmix_cut, err_c2_ngmix_cut])
    )

In [ ]:
g_corr_ngmix_local = {}
g_corr_ngmix_local = {}
R_shear_local = {}
R_selec_local = {}
ra_ngmix_local = {}
dec_ngmix_local = {}
c_ngmix_local = {}
c_err_ngmix_local = {}

# Additional mean and std of multiplicative bias
m = 0
dm = 0

# Loop over different calibration pixel sizes
for cal_pix in cal_pix_size_deg:
    npix_x = int(size_x_deg_new / cal_pix)
    npix_y = int(size_y_deg_new / cal_pix)
    if verbose:
        print(f'Calibration pixel size = {cal_pix} deg, number of pixels = {npix_x} x {npix_y}')
    
    # Call local_calibration with verbose=False to avoid too much std output
    (
        g_corr_ngmix_local[cal_pix],
        R_shear_local[cal_pix],
        R_selec_local[cal_pix],
        ra_ngmix_local[cal_pix],
        dec_ngmix_local[cal_pix],
        c_ngmix_local[cal_pix],
        c_err_ngmix_local[cal_pix]
    ) = local_calib(dd, x, y, min_x, max_x_new, min_y, max_y_new, npix_x, npix_y,
                    R_shear_ngmix, gal_metacal_ngmix.R_selection, m, dm, stats_file, verbose=False)

### Plots

#### Additive bias

In [ ]:
def sub_plot(nx, ny, i, im, title):

        plt.subplot(nx, ny, i)
        
        ax = plt.gca()
        im = ax.imshow(im)
        
        plt.title(title ,fontsize=fontsize)
        
        divider = make_axes_locatable(ax)
        
        plt.xticks(fontsize=fontsize)
        plt.yticks(fontsize=fontsize)

        cax = divider.append_axes("right", size="5%", pad=0.1)
        plt.colorbar(im, cax=cax)
        plt.yticks(fontsize=fontsize)
 
fontsize = 32
plt.figure(figsize=(74,45))

for i, cal_pix in enumerate(cal_pix_size_deg):

    for comp in (0, 1):
        subf = 4*i + 2*comp + 1
        
        sub_plot(5, 4, subf, c_ngmix_local[cal_pix][comp], rf'$c_{comp+1}$')
        sub_plot(5, 4, subf+1, c_err_ngmix_local[cal_pix][comp], rf'err $c_{comp+1}$')